# Банковский ИИ-ассистент: Prompting → RAG → Fine-Tuning (LoRA) → Гибрид

Практическая работа по адаптации LLM на примере ответов о тарифах и продуктах банка.

**Модель:** `Qwen/Qwen3-4B-Instruct-2507` (4-bit, T4 Colab)

**Репозиторий:** [llm-adaptation-banking-agent](https://github.com/pueraeternis/llm-adaptation-banking-agent)

## 0. Подготовка окружения (Colab)

В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.

In [ ]:
import os

if not os.path.exists("llm-adaptation-banking-agent"):
    print("Клонирование репозитория...")
    !git clone https://github.com/pueraeternis/llm-adaptation-banking-agent.git
    %cd llm-adaptation-banking-agent
else:
    print("Репозиторий уже на месте, переходим в каталог проекта.")
    %cd llm-adaptation-banking-agent

In [ ]:
print("Установка зависимостей...")
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

import numpy as np
import torch
import faiss
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

ROOT = Path(".").resolve()
for candidate in [Path("."), Path(".."), Path("/content/llm-adaptation-banking-agent")]:
    if (candidate / "data" / "banking_tariffs.txt").exists():
        ROOT = candidate.resolve()
        break

DATA_DIR = ROOT / "data"
TARIFFS_PATH = DATA_DIR / "banking_tariffs.txt"
LORA_DATASET_PATH = DATA_DIR / "lora_dataset.jsonl"

assert TARIFFS_PATH.exists(), f"Не найден {TARIFFS_PATH}"
assert LORA_DATASET_PATH.exists(), f"Не найден {LORA_DATASET_PATH}"

print("Корень проекта:", ROOT)
print("База знаний RAG:", TARIFFS_PATH)
print("LoRA датасет:", LORA_DATASET_PATH)
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Загрузка токенизатора: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Загрузка модели в 4-bit (это займёт 1–3 мин)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Модель и токенизатор готовы.")

In [ ]:
def generate_response(messages: list[dict], max_new_tokens: int = 256) -> str:
    """Генерация ответа chat-модели через apply_chat_template."""
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    print("Генерация ответа...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 1. Prompt Engineering

Базовый ответ модели с системным промптом без внешней базы знаний.

In [ ]:
SYSTEM_PROMPT = """Ты — вежливый банковский ассистент.
Отвечай только на вопросы о продуктах и тарифах банка.
Если не знаешь ответа — честно скажи, что уточнишь у специалиста.
Не придумывай цифры и условия."""

USER_QUESTION = "Сколько стоит обслуживание дебетовой карты «Стандарт»?"

prompt_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION},
]

print("Вопрос пользователя:", USER_QUESTION)
print("\n--- Ответ модели (Prompt Engineering) ---\n")
prompt_answer = generate_response(prompt_messages)
print(prompt_answer)

## 2. RAG (Retrieval-Augmented Generation)

Векторный поиск по `banking_tariffs.txt` (FAISS + `sentence-transformers`), топ-1 чанк в контекст промпта.

In [ ]:
EMBED_MODEL_ID = "paraphrase-multilingual-MiniLM-L12-v2"

print("Подготовка чанков из базы знаний...")
tariffs_text = TARIFFS_PATH.read_text(encoding="utf-8").strip()
raw_parts = tariffs_text.split("\n## ")
chunks = []
for part in raw_parts:
    part = part.strip()
    if not part:
        continue
    if not part.startswith("##"):
        part = "## " + part
    chunks.append(part)

print(f"Чанков в индексе: {len(chunks)}")
for i, ch in enumerate(chunks):
    title = ch.split("\n", 1)[0]
    print(f"  [{i}] {title}")

In [ ]:
print(f"Загрузка embedding-модели: {EMBED_MODEL_ID}...")
embed_model = SentenceTransformer(EMBED_MODEL_ID)

print("Векторизация чанков и построение FAISS IndexFlatL2...")
chunk_embeddings = embed_model.encode(chunks, convert_to_numpy=True).astype("float32")
dimension = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(chunk_embeddings)
print(f"Индекс FAISS построен: {faiss_index.ntotal} векторов, размерность {dimension}.")


def retrieve_top1(query: str) -> tuple[str, float]:
    query_embedding = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = faiss_index.search(query_embedding, 1)
    idx = int(indices[0][0])
    return chunks[idx], float(distances[0][0])

In [ ]:
rag_query = USER_QUESTION
print("Запрос для RAG:", rag_query)

top_chunk, distance = retrieve_top1(rag_query)
print(f"\nНайден релевантный чанк (L2={distance:.4f}):\n{top_chunk}\n")

rag_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            f"Контекст из базы знаний банка:\n{top_chunk}\n\n"
            f"Вопрос клиента: {rag_query}\n"
            "Ответь, опираясь только на контекст."
        ),
    },
]

print("--- Ответ модели (RAG) ---\n")
rag_answer = generate_response(rag_messages)
print(rag_answer)

## 3. Fine-Tuning (LoRA)

Дообучение адаптера на `lora_dataset.jsonl` (формат `messages`, JSON-ответы API-агента).

In [ ]:
print("Загрузка датасета для LoRA...")
train_dataset = load_dataset(
    "json",
    data_files=str(LORA_DATASET_PATH),
    split="train",
)
print(f"Примеров в датасете: {len(train_dataset)}")
print("\nПример диалога:")
for msg in train_dataset[0]["messages"]:
    preview = msg["content"][:120] + ("..." if len(msg["content"]) > 120 else "")
    print(f"  [{msg['role']}]: {preview}")

In [ ]:
def formatting_func(example: dict) -> str:
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
    )


print("Подготовка модели к k-bit обучению и подключение LoRA...")
model.train()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
sft_config = SFTConfig(
    output_dir="./lora_banking_agent",
    max_steps=15,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_steps=15,
    report_to="none",
    bf16=torch.cuda.is_available(),
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print("Старт обучения LoRA (15 шагов)...")
train_result = trainer.train()
if hasattr(train_result, "training_loss") and train_result.training_loss is not None:
    final_loss = train_result.training_loss
else:
    final_loss = trainer.state.log_history[-1].get("loss", float("nan"))
print(f"Обучение завершено. Loss: {final_loss:.4f}")

model.eval()

## 4. Гибрид (LoRA + RAG)

RAG находит актуальный тариф, дообученная LoRA-модель отвечает в JSON-формате API-агента.

In [ ]:
API_SYSTEM_PROMPT = (
    "Ты — банковский API-агент. Отвечай только в формате JSON "
    "с ключами 'response' и 'compliance_risk'."
)

HYBRID_QUESTION = (
    "Сколько стоит обслуживание карты Стандарт если я трачу 5000 рублей?"
)

print("Гибридный запрос:", HYBRID_QUESTION)
print("\nШаг 1: поиск контекста через FAISS (RAG)...")
hybrid_chunk, hybrid_distance = retrieve_top1(HYBRID_QUESTION)
print(f"Найден чанк (L2={hybrid_distance:.4f}):\n{hybrid_chunk}\n")

hybrid_messages = [
    {"role": "system", "content": API_SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            f"Контекст из базы знаний банка:\n{hybrid_chunk}\n\n"
            f"Вопрос клиента: {HYBRID_QUESTION}"
        ),
    },
]

print("Шаг 2: генерация JSON через дообученную LoRA-модель...")
hybrid_answer = generate_response(hybrid_messages, max_new_tokens=300)

print("\n--- Финальный ответ (LoRA + RAG) ---\n")
print(hybrid_answer)

## 5. Сравнение методов

| Метод | Плюсы | Минусы |
|-------|-------|--------|
| Prompt Engineering | Быстро, без обучения | Модель может галлюцинировать без фактов |
| RAG | Актуальные знания без переобучения | Зависит от качества поиска и базы |
| LoRA | Фиксированный JSON-формат ответа | Нужны данные и GPU; факты устаревают |
| **LoRA + RAG** | Точные факты + нужный формат | Сложнее пайплайн, два компонента |